# Regenerate Metrics And Reports

Rebuild aggregate metrics and confusion-matrix reports from saved 8-fold prediction CSVs.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

mplconfig_dir = ensure_dir(ROOT / '.matplotlib')
os.environ['MPLCONFIGDIR'] = str(mplconfig_dir.resolve())

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def summarize_prediction_frame(prediction_df: pd.DataFrame) -> dict[str, object]:
    y_true = prediction_df['true_label'].astype(str)
    y_pred = prediction_df['predicted_label'].astype(str)
    confusion = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'confusion_matrix': confusion,
    }


def save_confusion_matrix_png(confusion: np.ndarray, output_path: Path) -> None:
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        confusion,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        xticklabels=LABEL_ORDER,
        yticklabels=LABEL_ORDER,
        ax=axis,
    )
    axis.set_xlabel('Predicted')
    axis.set_ylabel('Actual')
    figure.tight_layout()
    figure.savefig(output_path, dpi=200)
    plt.close(figure)


def load_prediction_csvs_for_metrics(seed_metrics_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics_df = pd.read_csv(seed_metrics_path)
    prediction_frames: list[pd.DataFrame] = []
    for row in metrics_df.to_dict(orient='records'):
        prediction_path = Path(str(row['predictions_path']))
        prediction_df = pd.read_csv(prediction_path)
        prediction_df['fold'] = str(row.get('fold', ''))
        prediction_df['seed'] = int(row.get('seed', 0))
        prediction_frames.append(prediction_df)
    if not prediction_frames:
        raise ValueError(f'No prediction CSVs were found from {seed_metrics_path}')
    return metrics_df, pd.concat(prediction_frames, ignore_index=True)


def regenerate_official_reports(seed_metrics_path: Path, output_root: Path) -> dict[str, Path]:
    metrics_df, all_predictions_df = load_prediction_csvs_for_metrics(seed_metrics_path)
    summary_rows: list[dict[str, object]] = []
    for (fold, seed), group_df in all_predictions_df.groupby(['fold', 'seed']):
        summary = summarize_prediction_frame(group_df)
        summary_rows.append(
            {
                'fold': fold,
                'seed': seed,
                'accuracy': summary['accuracy'],
                'macro_precision': summary['macro_precision'],
                'macro_recall': summary['macro_recall'],
                'macro_f1': summary['macro_f1'],
                'prediction_count': len(group_df),
            }
        )

    fold_summary_df = pd.DataFrame(summary_rows).sort_values(['fold', 'seed']).reset_index(drop=True)
    prediction_distribution_df = (
        all_predictions_df.groupby(['predicted_label']).size().rename('count').reset_index().sort_values('predicted_label')
    )
    overall_summary = summarize_prediction_frame(all_predictions_df)
    overall_confusion_df = pd.DataFrame(overall_summary['confusion_matrix'], index=LABEL_ORDER, columns=LABEL_ORDER)

    fold_summary_path = output_root / 'processed_roi8_cnn_only_fold_summary.csv'
    prediction_distribution_path = output_root / 'processed_roi8_cnn_only_prediction_distribution.csv'
    overall_confusion_csv_path = output_root / 'processed_roi8_cnn_only_overall_confusion_matrix.csv'
    overall_confusion_png_path = output_root / 'processed_roi8_cnn_only_overall_confusion_matrix.png'

    fold_summary_df.to_csv(fold_summary_path, index=False)
    prediction_distribution_df.to_csv(prediction_distribution_path, index=False)
    overall_confusion_df.to_csv(overall_confusion_csv_path)
    save_confusion_matrix_png(overall_summary['confusion_matrix'], overall_confusion_png_path)

    return {
        'fold_summary': fold_summary_path,
        'prediction_distribution': prediction_distribution_path,
        'overall_confusion_csv': overall_confusion_csv_path,
        'overall_confusion_png': overall_confusion_png_path,
    }


In [ ]:
OUTPUT_ROOT = ensure_dir(Path(str(override('EIGHTFOLD_OUTPUT_ROOT', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8fold_processed_roi_cnn_only'))))
SEED_METRICS_PATH = Path(str(override('SEED_METRICS_PATH', OUTPUT_ROOT / 'processed_roi8_cnn_only_seed_metrics.csv')))

written_paths = regenerate_official_reports(SEED_METRICS_PATH, OUTPUT_ROOT)
print('\n'.join(f'{name}: {path}' for name, path in written_paths.items()))
